> Repository edition: run from the repository root using an environment with the dependencies in `requirements.txt`. Machine-specific environment paths have been replaced. Historical outputs are retained except environment checks; they are not evidence of a fresh rerun. See `docs/REPRODUCIBILITY.md`.


In [1]:
import numpy as np


# Load the initial input data for Function 1
# Each row represents one previously evaluated query point
inputs = np.load("initial_data/function_1/initial_inputs.npy")

# Load the corresponding output values for Function 1
# Each output corresponds to the input point in the same row
outputs = np.load("initial_data/function_1/initial_outputs.npy")

# Check the shape of the loaded data
# Function 1 should contain 10 observations in a 2-dimensional input space
print("Input shape:", inputs.shape)
print("Output shape:", outputs.shape)

Input shape: (10, 2)
Output shape: (10,)


In [2]:
# Display the initial query points
print("Inputs:")
print(inputs)

# Display the corresponding objective function values
print("\nOutputs:")
print(outputs)

# Identify the best output observed so far
best_index = np.argmax(outputs)
best_output = outputs[best_index]
best_input = inputs[best_index]

# Print the current best observation
print("\nBest output observed so far:", best_output)
print("Input corresponding to the best output:", best_input)

Inputs:
[[0.31940389 0.76295937]
 [0.57432921 0.8798981 ]
 [0.73102363 0.73299988]
 [0.84035342 0.26473161]
 [0.65011406 0.68152635]
 [0.41043714 0.1475543 ]
 [0.31269116 0.07872278]
 [0.68341817 0.86105746]
 [0.08250725 0.40348751]
 [0.88388983 0.58225397]]

Outputs:
[ 1.32267704e-079  1.03307824e-046  7.71087511e-016  3.34177101e-124
 -3.60606264e-003 -2.15924904e-054 -2.08909327e-091  2.53500115e-040
  3.60677119e-081  6.22985647e-048]

Best output observed so far: 7.710875114502849e-16
Input corresponding to the best output: [0.73102363 0.73299988]


In [3]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

# Define and fit a Gaussian Process surrogate model
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(length_scale=[0.2, 0.2], length_scale_bounds=(0.02, 2.0), nu=2.5)
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-10, 1e-1))
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

gp.fit(inputs, outputs)

print("Fitted kernel:", gp.kernel_)

Fitted kernel: 1.02**2 * Matern(length_scale=[2, 0.0264], nu=2.5) + WhiteKernel(noise_level=2.64e-08)


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [4]:
# Generate possible new query points within the valid search space
rng = np.random.default_rng(42)
candidates = rng.uniform(0, 0.999999, size=(100000, 2))

# Predict the expected output and uncertainty for every candidate
mean, std = gp.predict(candidates, return_std=True)

# Upper Confidence Bound acquisition function
# It balances high predicted outputs and high uncertainty
beta = 2.0
ucb = mean + beta * std

# Select the point with the highest UCB score
best_candidate_index = np.argmax(ucb)
next_query = candidates[best_candidate_index]

print("Suggested next query:", next_query)
print(f"Portal format: {next_query[0]:.6f}-{next_query[1]:.6f}")


Suggested next query: [0.61612314 0.20742286]
Portal format: 0.616123-0.207423


In [5]:
#Load the initial input data for Function 2
inputs_2 = np.load("initial_data/function_2/initial_inputs.npy")
outputs_2 = np.load("initial_data/function_2/initial_outputs.npy")


# Check the dimensions of the loaded data for Function 2
print("Input shape for Function 2:", inputs_2.shape)
print("Output shape for Function 2:", outputs_2.shape)

#Identify the best observations so far
best_index_2 = np.argmax(outputs_2)

print("Best input:", inputs_2[best_index_2])
print("Best output:", outputs_2[best_index_2])

Input shape for Function 2: (10, 2)
Output shape for Function 2: (10,)
Best input: [0.70263656 0.9265642 ]
Best output: 0.6112052157614438


In [6]:
# Define a separate kernel for Function 2
kernel_2 = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=[0.2, 0.2],
        length_scale_bounds=(0.02, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-1)
    )
)

# Create the Gaussian Process model for Function 2
gp_2 = GaussianProcessRegressor(
    kernel=kernel_2,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

# Fit the model using Function 2 data
gp_2.fit(inputs_2, outputs_2)

print("Fitted kernel for Function 2:", gp_2.kernel_)

Fitted kernel for Function 2: 0.984**2 * Matern(length_scale=[0.0601, 2], nu=2.5) + WhiteKernel(noise_level=0.0123)


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [7]:
# Generate candidate points for Function 2
rng_2 = np.random.default_rng(42)
candidates_2 = rng_2.uniform(0, 0.999999, size=(100000, 2))

# Estimate outputs and uncertainty using the Function 2 model
mean_2, std_2 = gp_2.predict(candidates_2, return_std=True)

# Balance exploitation and exploration using UCB
beta_2 = 2.0
ucb_2 = mean_2 + beta_2 * std_2

# Select the candidate with the highest UCB score
best_candidate_index_2 = np.argmax(ucb_2)
next_query_2 = candidates_2[best_candidate_index_2]

# Display the proposed point and the required submission format
print("Suggested next query for Function 2:", next_query_2)
print(f"Portal format: {next_query_2[0]:.6f}-{next_query_2[1]:.6f}")

Suggested next query for Function 2: [0.72069042 0.00372188]
Portal format: 0.720690-0.003722


In [9]:
#Load the initial input data for Function 3
inputs_3 = np.load("initial_data/function_3/initial_inputs.npy")
outputs_3 = np.load("initial_data/function_3/initial_outputs.npy")


# Check the dimensions of the loaded data for Function 3
print("Input shape for Function 3:", inputs_3.shape)
print("Output shape for Function 3:", outputs_3.shape)

#Identify the best observations so far
best_index_3 = np.argmax(outputs_3)

print("Best input:", inputs_3[best_index_3])
print("Best output:", outputs_3[best_index_3])

Input shape for Function 3: (15, 3)
Output shape for Function 3: (15,)
Best input: [0.49258141 0.61159319 0.34017639]
Best output: -0.034835313350078584


In [11]:
# Define a separate kernel for Function 3
kernel_3 = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=[0.2, 0.2, 0.2],
        length_scale_bounds=(0.02, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-1)
    )
)

# Create the Gaussian Process model for Function 3
gp_3 = GaussianProcessRegressor(
    kernel=kernel_3,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

# Fit the model using all 15 observations
gp_3.fit(inputs_3, outputs_3)

print("Fitted kernel for Function 3:", gp_3.kernel_)

Fitted kernel for Function 3: 1.16**2 * Matern(length_scale=[2, 1.76, 0.0615], nu=2.5) + WhiteKernel(noise_level=7.92e-07)


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [12]:
# Generate candidate points with three input variables
rng_3 = np.random.default_rng(42)
candidates_3 = rng_3.uniform(0, 0.999999, size=(100000, 3))

# Predict outputs and uncertainty
mean_3, std_3 = gp_3.predict(candidates_3, return_std=True)

# Calculate the UCB score
beta_3 = 2.0
ucb_3 = mean_3 + beta_3 * std_3

# Select the candidate with the highest score
best_candidate_index_3 = np.argmax(ucb_3)
next_query_3 = candidates_3[best_candidate_index_3]

# Format the three coordinates for the portal
print("Suggested next query for Function 3:", next_query_3)
print("Portal format:", "-".join(f"{x:.6f}" for x in next_query_3))


Suggested next query for Function 3: [5.94297042e-02 4.71717644e-02 2.26996786e-05]
Portal format: 0.059430-0.047172-0.000023


In [13]:
#Load the initial input data for Function 4
inputs_4 = np.load("initial_data/function_4/initial_inputs.npy")
outputs_4 = np.load("initial_data/function_4/initial_outputs.npy")


# Check the dimensions of the loaded data for Function 4
print("Input shape for Function 4:", inputs_4.shape)
print("Output shape for Function 4:", outputs_4.shape)

#Identify the best observations so far
best_index_4 = np.argmax(outputs_4)

print("Best input:", inputs_4[best_index_4])
print("Best output:", outputs_4[best_index_4])

Input shape for Function 4: (30, 4)
Output shape for Function 4: (30,)
Best input: [0.57776561 0.42877174 0.42582587 0.24900741]
Best output: -4.025542281908162


In [15]:
# Define a separate kernel for Function 4
kernel_4 = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=[0.2, 0.2, 0.2, 0.2],
        length_scale_bounds=(0.02, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-1)
    )
)

# Create the Gaussian Process model for Function 4
gp_4 = GaussianProcessRegressor(
    kernel=kernel_4,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

# Fit the model using all 30 observations
gp_4.fit(inputs_4, outputs_4)

print("Fitted kernel for Function 4:", gp_4.kernel_)

Fitted kernel for Function 4: 4.11**2 * Matern(length_scale=[1.83, 1.62, 1.84, 1.78], nu=2.5) + WhiteKernel(noise_level=1e-10)


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [16]:
# Generate candidate points with four input variables
rng_4 = np.random.default_rng(42)
candidates_4 = rng_4.uniform(0, 0.999999, size=(100000, 4))

# Predict outputs and uncertainty
mean_4, std_4 = gp_4.predict(candidates_4, return_std=True)

# Balance exploitation and exploration using UCB
beta_4 = 2.0
ucb_4 = mean_4 + beta_4 * std_4

# Select the candidate with the highest UCB score
best_candidate_index_4 = np.argmax(ucb_4)
next_query_4 = candidates_4[best_candidate_index_4]

# Display the proposed point and format it for the portal
print("Suggested next query for Function 4:", next_query_4)
print("Portal format:", "-".join(f"{x:.6f}" for x in next_query_4))


Suggested next query for Function 4: [0.404204   0.42014894 0.3561527  0.44923003]
Portal format: 0.404204-0.420149-0.356153-0.449230


In [17]:
# Load the initial data for Function 5
inputs_5 = np.load("initial_data/function_5/initial_inputs.npy")
outputs_5 = np.load("initial_data/function_5/initial_outputs.npy")

# Check the dimensions
print("Input shape for Function 5:", inputs_5.shape)
print("Output shape for Function 5:", outputs_5.shape)

# Identify the best observation so far
best_index_5 = np.argmax(outputs_5)

print("Best input:", inputs_5[best_index_5])
print("Best output:", outputs_5[best_index_5])


Input shape for Function 5: (20, 4)
Output shape for Function 5: (20,)
Best input: [0.22418902 0.84648049 0.87948418 0.87851568]
Best output: 1088.8596181962705


In [18]:
# Define a separate kernel for Function 5
kernel_5 = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=[0.2, 0.2, 0.2, 0.2],
        length_scale_bounds=(0.02, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-1)
    )
)

# Create the Gaussian Process model for Function 5
gp_5 = GaussianProcessRegressor(
    kernel=kernel_5,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

# Fit the model using all 20 observations
gp_5.fit(inputs_5, outputs_5)

print("Fitted kernel for Function 5:", gp_5.kernel_)

Fitted kernel for Function 5: 1.35**2 * Matern(length_scale=[2, 2, 0.463, 0.297], nu=2.5) + WhiteKernel(noise_level=2.62e-10)


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [19]:
# Generate candidate points with four input variables
rng_5 = np.random.default_rng(42)
candidates_5 = rng_5.uniform(0, 0.999999, size=(100000, 4))

# Predict outputs and uncertainty
mean_5, std_5 = gp_5.predict(candidates_5, return_std=True)

# Balance exploitation and exploration using UCB
beta_5 = 2.0
ucb_5 = mean_5 + beta_5 * std_5

# Select the candidate with the highest UCB score
best_candidate_index_5 = np.argmax(ucb_5)
next_query_5 = candidates_5[best_candidate_index_5]

# Display the proposed point and format it for the portal
print("Suggested next query for Function 5:", next_query_5)
print("Portal format:", "-".join(f"{x:.6f}" for x in next_query_5))

Suggested next query for Function 5: [0.29971915 0.81479338 0.99005694 0.99869102]
Portal format: 0.299719-0.814793-0.990057-0.998691


In [20]:
# Load the initial data for Function 6
inputs_6 = np.load("initial_data/function_6/initial_inputs.npy")
outputs_6 = np.load("initial_data/function_6/initial_outputs.npy")

# Check the dimensions
print("Input shape for Function 6:", inputs_6.shape)
print("Output shape for Function 6:", outputs_6.shape)

# Identify the best observation so far
best_index_6 = np.argmax(outputs_6)

print("Best input:", inputs_6[best_index_6])
print("Best output:", outputs_6[best_index_6])

Input shape for Function 6: (20, 5)
Output shape for Function 6: (20,)
Best input: [0.7281861  0.15469257 0.73255167 0.69399651 0.05640131]
Best output: -0.7142649478202404


In [21]:
# Define a separate kernel for Function 6
kernel_6 = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=[0.2, 0.2, 0.2, 0.2, 0.2],
        length_scale_bounds=(0.02, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-1)
    )
)

# Create the Gaussian Process model for Function 6
gp_6 = GaussianProcessRegressor(
    kernel=kernel_6,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

# Fit the model using all 20 observations
gp_6.fit(inputs_6, outputs_6)

print("Fitted kernel for Function 6:", gp_6.kernel_)

Fitted kernel for Function 6: 2.33**2 * Matern(length_scale=[0.827, 1.57, 1.95, 2, 2], nu=2.5) + WhiteKernel(noise_level=0.0446)


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [22]:
# Generate candidate points with five input variables
rng_6 = np.random.default_rng(42)
candidates_6 = rng_6.uniform(0, 0.999999, size=(100000, 5))

# Predict outputs and uncertainty
mean_6, std_6 = gp_6.predict(candidates_6, return_std=True)

# Balance exploitation and exploration using UCB
beta_6 = 2.0
ucb_6 = mean_6 + beta_6 * std_6

# Select the candidate with the highest UCB score
best_candidate_index_6 = np.argmax(ucb_6)
next_query_6 = candidates_6[best_candidate_index_6]

# Display the proposed point and format it for the portal
print("Suggested next query for Function 6:", next_query_6)
print("Portal format:", "-".join(f"{x:.6f}" for x in next_query_6))

Suggested next query for Function 6: [0.40231416 0.05291819 0.98235581 0.99110783 0.0041613 ]
Portal format: 0.402314-0.052918-0.982356-0.991108-0.004161


In [23]:
# Load the initial data for Function 7
inputs_7 = np.load("initial_data/function_7/initial_inputs.npy")
outputs_7 = np.load("initial_data/function_7/initial_outputs.npy")

# Check the dimensions
print("Input shape for Function 7:", inputs_7.shape)
print("Output shape for Function 7:", outputs_7.shape)

# Identify the best observation so far
best_index_7 = np.argmax(outputs_7)

print("Best input:", inputs_7[best_index_7])
print("Best output:", outputs_7[best_index_7])

Input shape for Function 7: (30, 6)
Output shape for Function 7: (30,)
Best input: [0.05789554 0.49167222 0.24742222 0.21811844 0.42042833 0.73096984]
Best output: 1.3649683044991994


In [24]:
# Define a separate kernel for Function 7
kernel_7 = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=[0.2, 0.2, 0.2, 0.2, 0.2, 0.2],
        length_scale_bounds=(0.02, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-1)
    )
)

# Create the Gaussian Process model for Function 7
gp_7 = GaussianProcessRegressor(
    kernel=kernel_7,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

# Fit the model using all 30 observations
gp_7.fit(inputs_7, outputs_7)

print("Fitted kernel for Function 7:", gp_7.kernel_)


Fitted kernel for Function 7: 1.02**2 * Matern(length_scale=[1.03, 2, 2, 0.383, 0.194, 0.235], nu=2.5) + WhiteKernel(noise_level=3.43e-09)


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [25]:
# Generate candidate points with six input variables
rng_7 = np.random.default_rng(42)
candidates_7 = rng_7.uniform(0, 0.999999, size=(100000, 6))

# Predict outputs and uncertainty
mean_7, std_7 = gp_7.predict(candidates_7, return_std=True)

# Balance exploitation and exploration using UCB
beta_7 = 2.0
ucb_7 = mean_7 + beta_7 * std_7

# Select the candidate with the highest UCB score
best_candidate_index_7 = np.argmax(ucb_7)
next_query_7 = candidates_7[best_candidate_index_7]

# Display the proposed point and format it for the portal
print("Suggested next query for Function 7:", next_query_7)
print("Portal format:", "-".join(f"{x:.6f}" for x in next_query_7))

Suggested next query for Function 7: [0.01351687 0.26708543 0.54500343 0.14052587 0.36730412 0.76512138]
Portal format: 0.013517-0.267085-0.545003-0.140526-0.367304-0.765121


In [26]:
# Load the initial data for Function 8
inputs_8 = np.load("initial_data/function_8/initial_inputs.npy")
outputs_8 = np.load("initial_data/function_8/initial_outputs.npy")

# Check the dimensions
print("Input shape for Function 8:", inputs_8.shape)
print("Output shape for Function 8:", outputs_8.shape)

# Identify the best observation so far
best_index_8 = np.argmax(outputs_8)

print("Best input:", inputs_8[best_index_8])
print("Best output:", outputs_8[best_index_8])

Input shape for Function 8: (40, 8)
Output shape for Function 8: (40,)
Best input: [0.05644741 0.06595555 0.02292868 0.03878647 0.40393544 0.80105533
 0.48830701 0.89308498]
Best output: 9.598482002566342


In [27]:
# Define a separate kernel for Function 8
kernel_8 = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=[0.2] * 8,
        length_scale_bounds=(0.02, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-1)
    )
)

# Create the Gaussian Process model for Function 8
gp_8 = GaussianProcessRegressor(
    kernel=kernel_8,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

# Fit the model using all 40 observations
gp_8.fit(inputs_8, outputs_8)

print("Fitted kernel for Function 8:", gp_8.kernel_)

Fitted kernel for Function 8: 1.11**2 * Matern(length_scale=[1.18, 2, 0.914, 2, 2, 2, 1.24, 2], nu=2.5) + WhiteKernel(noise_level=1.55e-10)


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-pack

In [28]:
# Generate candidate points with eight input variables
rng_8 = np.random.default_rng(42)
candidates_8 = rng_8.uniform(0, 0.999999, size=(100000, 8))

# Predict outputs and uncertainty
mean_8, std_8 = gp_8.predict(candidates_8, return_std=True)

# Balance exploitation and exploration using UCB
beta_8 = 2.0
ucb_8 = mean_8 + beta_8 * std_8

# Select the candidate with the highest UCB score
best_candidate_index_8 = np.argmax(ucb_8)
next_query_8 = candidates_8[best_candidate_index_8]

# Display the proposed point and format it for the portal
print("Suggested next query for Function 8:", next_query_8)
print("Portal format:", "-".join(f"{x:.6f}" for x in next_query_8))

Suggested next query for Function 8: [0.05922956 0.02456981 0.03743642 0.07199579 0.9553429  0.14538764
 0.13540701 0.53911253]
Portal format: 0.059230-0.024570-0.037436-0.071996-0.955343-0.145388-0.135407-0.539113
